In [0]:

import sys
from pathlib import Path

print("Python funcionando!")
print("Versão:", sys.version)
print("Diretório atual:", Path.cwd())


In [0]:

from pathlib import Path

# Caminho do documento
PDF_PATH = Path.cwd() / "docs" / "caderno_cidadania_financeira.pdf"

# Verificar se o arquivo existe
print("Caminho:", PDF_PATH)
print("Arquivo encontrado:", PDF_PATH.is_file())

if PDF_PATH.is_file():
    tamanho_mb = PDF_PATH.stat().st_size / (1024 * 1024)
    print(f"Tamanho: {tamanho_mb:.2f} MB")
else:
    print("Erro: documento não encontrado.")


In [0]:

%pip install pypdf
#A biblioteca pypdf permite ler documentos PDF e extrair o texto de suas páginas

In [0]:

from pypdf import PdfReader

# Abrir o documento
reader = PdfReader(str(PDF_PATH))

# Armazenar texto e número de cada página
paginas = []

for numero, pagina in enumerate(reader.pages, start=1):
    texto = pagina.extract_text() or ""

    if texto.strip():
        paginas.append({
            "pagina": numero,
            "texto": texto.strip()
        })

# Exibir estatísticas
print("Total de páginas:", len(reader.pages))
print("Páginas com texto:", len(paginas))
print("Caracteres extraídos:", sum(
    len(p["texto"]) for p in paginas
))

# Visualizar um exemplo
print("\nEXEMPLO DE TEXTO EXTRAÍDO:\n")
print(paginas[0]["texto"][:1000])


## Etapa 6. Chunking, divisão do documento em trechos

In [0]:

%pip install -q langchain-text-splitters
#Pacote independente do ecossistema LangChain projetado especificamente para dividir textos extensos em partes menores e gerenciáveis (chamadas de chunks).


In [0]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuração inicial do chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    length_function=len,
    add_start_index=True
)

# Dividir o texto, preservando as páginas
chunks = []

for pagina in paginas:
    trechos = splitter.split_text(pagina["texto"])

    for indice, trecho in enumerate(trechos, start=1):
        chunks.append({
            "id": f"bcb-p{pagina['pagina']:03d}-c{indice:03d}",
            "pagina": pagina["pagina"],
            "fonte": "Caderno de Educação Financeira - BCB (2026)",
            "texto": trecho
        })

# Estatísticas
print("Total de chunks:", len(chunks))
print("Total de páginas processadas:", len(paginas))
print("Tamanho médio dos chunks:",
      round(sum(len(c["texto"]) for c in chunks) / len(chunks)))

# Exemplo
print("\nEXEMPLO DE CHUNK:\n")
print(chunks[10])


## Etapa 7. Gerar embeddings

Agora vamos transformar os textos em representações numéricas para que o RAG consiga encontrar trechos semanticamente relacionados a uma pergunta.
Suporta português, gera vetores de 384 dimensões e pode ser executado localmente, sem precisar de uma API paga.

In [0]:

%pip install -q sentence-transformers
#Transforma frases, parágrafos e documentos inteiros em representações numéricas chamadas de embeddings (vetores)

In [0]:

from sentence_transformers import SentenceTransformer

# Modelo de embeddings multilíngue
NOME_MODELO = "intfloat/multilingual-e5-small"

modelo_embeddings = SentenceTransformer(NOME_MODELO)

# Selecionar três chunks para teste
textos_teste = [
    "passage: " + chunk["texto"]
    for chunk in chunks[:3]
]

# Gerar os vetores
vetores_teste = modelo_embeddings.encode(
    textos_teste,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Modelo:", NOME_MODELO)
print("Chunks processados:", len(vetores_teste))
print("Dimensões:", vetores_teste.shape)

print("\nPrimeiros valores do vetor:")
print(vetores_teste[0][:8])


## Etapa 8. Criar o banco vetorial com ChromaDB

In [0]:

%pip install -q chromadb
# Instalação do ChromaDB
#O ChromaDB permite armazenar embeddings, documentos e metadados, além de executar consultas por similaridade.


In [0]:

import chromadb

# 1. Preparar os textos
textos = [
    "passage: " + chunk["texto"]
    for chunk in chunks
]

# 2. Gerar embeddings dos 290 chunks
vetores = modelo_embeddings.encode(
    textos,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings gerados:", vetores.shape)

# 3. Inicializar o ChromaDB em memória
cliente_chroma = chromadb.Client()

colecao = cliente_chroma.get_or_create_collection(
    name="poc_educacao_financeira",
    embedding_function=None,
    configuration={
        "hnsw": {"space": "cosine"}
    }
)

# 4. Armazenar textos, embeddings e metadados
colecao.upsert(
    ids=[c["id"] for c in chunks],
    embeddings=vetores.tolist(),
    documents=[c["texto"] for c in chunks],
    metadatas=[
        {
            "pagina": c["pagina"],
            "fonte": c["fonte"]
        }
        for c in chunks
    ]
)

# 5. Verificar o banco
print("Documentos indexados:", colecao.count())
print("Banco vetorial pronto!")


## Etapa 9. Implementar a busca semântica

In [0]:

# Pergunta para testar a recuperação
pergunta = (
    "Como uma pessoa pode se preparar financeiramente "
    "para perder sua principal fonte de renda?"
)

# 1. Transformar a pergunta em embedding
vetor_pergunta = modelo_embeddings.encode(
    "query: " + pergunta,
    normalize_embeddings=True
)

# 2. Buscar os 5 chunks mais próximos
resultados = colecao.query(
    query_embeddings=[vetor_pergunta.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

# 3. Exibir os resultados
print("PERGUNTA:", pergunta)
print("\nTRECHOS RECUPERADOS:\n")

for i, (texto, metadata, distancia) in enumerate(
    zip(
        resultados["documents"][0],
        resultados["metadatas"][0],
        resultados["distances"][0]
    ),
    start=1
):
    print(f"========== RESULTADO {i} ==========")
    print(f"Página do PDF: {metadata['pagina']}")
    print(f"Distância: {distancia:.4f}")
    print(f"Fonte: {metadata['fonte']}")
    print(f"Texto:\n{texto[:600]}")
    print()


## Etapa 10. Testar uma pergunta mais específica

In [0]:

# Nova pergunta, mais específica
pergunta_teste = (
    "O que fazer diante da perda de emprego "
    "ou redução da renda familiar? "
    "Qual a importância da reserva de emergência?"
)

# Gerar embedding da pergunta
vetor_teste = modelo_embeddings.encode(
    "query: " + pergunta_teste,
    normalize_embeddings=True
)

# Recuperar os 8 resultados mais próximos
resultados_teste = colecao.query(
    query_embeddings=[vetor_teste.tolist()],
    n_results=8,
    include=["documents", "metadatas", "distances"]
)

print("PERGUNTA:", pergunta_teste)

for i, (texto, metadata, distancia) in enumerate(
    zip(
        resultados_teste["documents"][0],
        resultados_teste["metadatas"][0],
        resultados_teste["distances"][0]
    ),
    start=1
):
    print(f"\nRESULTADO {i}")
    print(f"Página: {metadata['pagina']}")
    print(f"Distância: {distancia:.4f}")
    print(f"Texto: {texto[:400]}")


## Etapa 11. Escolher o LLM para gerar as respostas

> Modelo escolhido: GPT OSS 20B

## Etapa 12. Conectar o LLM ao notebook

In [0]:

%pip install -q databricks-openai
# A parceria estratégica e a integração nativa entre a Databricks e a OpenAI (anunciada e expandida em acordos de alto impacto) levam os modelos avançados de Inteligência Artificial da OpenAI diretamente para dentro da plataforma de dados e IA da Databricks


In [0]:

from databricks_openai import DatabricksOpenAI

# Autenticação automática no Databricks
client = DatabricksOpenAI()

# Teste inicial do modelo
resposta = client.chat.completions.create(
    model="databricks-gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": "Você é um assistente de educação financeira. Responda em português do Brasil, de forma clara e objetiva."
        },
        {
            "role": "user",
            "content": "O que é uma reserva de emergência? Responda em duas frases."
        }
    ],
    max_tokens=500
)

print(resposta.choices[0].message.content)


## Etapa 13. Construir o RAG completo

![image_1794240735706.png](./image_1794240735706.png "image_1794240735706.png")

In [0]:

# ==========================================
# ETAPA 13 - RAG COMPLETO
# ==========================================

# 1. Pergunta do usuário
pergunta_rag = (
    "Como uma pessoa pode se preparar financeiramente "
    "para perder sua principal fonte de renda?"
)

# 2. Gerar o embedding da pergunta
vetor_pergunta = modelo_embeddings.encode(
    "query: " + pergunta_rag,
    normalize_embeddings=True
)

# 3. Recuperar os 5 chunks mais relevantes
resultados_rag = colecao.query(
    query_embeddings=[vetor_pergunta.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

# 4. Construir o contexto com referências
trechos = []

for i, (texto, metadata) in enumerate(
    zip(
        resultados_rag["documents"][0],
        resultados_rag["metadatas"][0]
    ),
    start=1
):
    trechos.append(
        f"[TRECHO {i} | Página {metadata['pagina']} do PDF]\n"
        f"{texto}"
    )

if not trechos:
    raise ValueError("Nenhum trecho foi recuperado.")

contexto = "\n\n".join(trechos)

# 5. Instruções para o LLM
system_prompt = """
Você é um assistente de educação financeira.

Responda em português do Brasil, utilizando
exclusivamente as informações do contexto fornecido.

Regras:
1. Não invente informações.
2. Não utilize conhecimentos externos.
3. Fundamente as afirmações nos trechos recuperados.
4. Cite as páginas do PDF que sustentam a resposta.
5. Utilize referências como [Página 42 do PDF].
6. Se não houver informações suficientes,
   declare explicitamente essa limitação.
7. Não apresente recomendações como se fossem
   do Banco Central sem evidência no contexto.
"""

# 6. Enviar pergunta e contexto ao LLM
resposta_rag = client.chat.completions.create(
    model="databricks-gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"CONTEXTO:\n{contexto}\n\n"
                f"PERGUNTA:\n{pergunta_rag}"
            )
        }
    ],
    max_tokens=1200
)

# 7. Exibir resposta
print("PERGUNTA:")
print(pergunta_rag)

print("\nRESPOSTA DO RAG:\n")
print(resposta_rag.choices[0].message.content)

# 8. Mostrar as fontes recuperadas
print("\nPÁGINAS RECUPERADAS:")
for metadata in resultados_rag["metadatas"][0]:
    print(f"- Página {metadata['pagina']} do PDF")


In [0]:
Etapa 14. Avaliação quantitativa do RAG

In [0]:
# ============================================================
# ETAPA 14 — AVALIAÇÃO DO RAG
# ============================================================
# Avaliamos o RAG em três eixos:
#   1. RECUPERAÇÃO  — Recall@K, Precision@K, MRR, NDCG
#   2. GERAÇÃO      — BLEU-4 e ROUGE-L (vs. resposta de referência)
#   3. LLM-AS-A-JUDGE — groundedness, relevance, factual consistency
#
# O que cada métrica mede:
#   Recall@K     → das páginas relevantes (ground truth), quantas
#                  aparecem entre os K chunks recuperados.
#   Precision@K  → dos K chunks recuperados, quantos vêm de uma
#                  página relevante.
#   MRR          → 1/posição do primeiro chunk relevante.
#   NDCG         → qualidade do ranking: relevantes no topo valem
#                  mais; cada página relevante pontua uma única vez.
#   BLEU-4       → sobreposição de n-gramas (1 a 4) com a referência.
#   ROUGE-L      → maior subsequência comum (LCS) com a referência.
#   LLM-Judge    → o LLM atribui nota 0-5 a groundedness (aderência),
#                  relevance (relevância) e factual consistency.
# ============================================================

import math
import numpy as np

# ----------------------------------------------------------------
# 14.1  DATASET DE TESTE (ground truth)
# ----------------------------------------------------------------
# Cada item contém:
#   pergunta            → consulta do usuário
#   paginas_relevantes  → páginas do PDF que contêm a resposta
#                         (anotadas manualmente — ground truth)
#   resposta_referencia → resposta-modelo escrita a partir do PDF
# ----------------------------------------------------------------

dataset_avaliacao = [
    {
        "pergunta": "Como uma pessoa pode se preparar financeiramente para perder sua principal fonte de renda?",
        "paginas_relevantes": {28, 35, 42, 79},
        "resposta_referencia": (
            "Para se preparar, a pessoa deve manter um orçamento superavitário, "
            "separando parte da renda para poupar antes de pagar despesas. "
            "É recomendado automatizar aportes de poupança, diferenciar "
            "necessidades de desejos e formar uma reserva de emergência. "
            "Na perda de emprego, deve-se reduzir despesas proporcionalmente "
            "à queda de renda para evitar endividamento excessivo."
        ),
    },
    {
        "pergunta": "O que são juros compostos e como funcionam?",
        "paginas_relevantes": {33, 34},
        "resposta_referencia": (
            "Juros compostos são calculados sobre o capital acrescido dos "
            "juros acumulados, ou seja, juros sobre juros. No exemplo do "
            "caderno, R$1.000 a 5% ao mês passam para R$1.050 no primeiro mês, "
            "R$1.102,50 no segundo, e assim por diante. Quanto maior o prazo, "
            "maior o efeito dos juros compostos, beneficiando quem poupa cedo."
        ),
    },
    {
        "pergunta": "Como planejar a aposentadoria do ponto de vista financeiro?",
        "paginas_relevantes": {74, 75, 76, 77},
        "resposta_referencia": (
            "O planejamento da aposentadoria deve considerar a incerteza do "
            "futuro e o aumento da expectativa de vida. Recomenda-se iniciar "
            "a reserva para aposentadoria assim que se começa a trabalhar, "
            "poupando parte do salário. É importante conhecer os regimes de "
            "previdência obrigatórios e complementares, seus direitos e "
            "deveres, e investir de forma regular para garantir uma renda "
            "estável na terceira idade."
        ),
    },
    {
        "pergunta": "Quais cuidados devo ter ao contratar um seguro?",
        "paginas_relevantes": {73, 77},
        "resposta_referencia": (
            "Ao contratar um seguro, é importante comparar preços, pois pode "
            "haver grande variação entre seguradoras. Deve-se entender as "
            "coberturas e exclusões, avaliar as necessidades reais, verificar "
            "a idoneidade da seguradora e ler o contrato com atenção antes "
            "de assinar."
        ),
    },
    {
        "pergunta": "Qual a importância de poupar e como começar?",
        "paginas_relevantes": {28, 29, 34},
        "resposta_referencia": (
            "Poupar é fundamental para realizar sonhos, prevenir imprevistos "
            "e construir segurança financeira. Para começar, recomenda-se "
            "pagar a si mesmo primeiro, separando parte da renda antes das "
            "despesas, automatizar investimentos em datas predefinidas e "
            "poupar de forma regular, mesmo que com valores pequenos. "
            "Quanto mais cedo começar, maior o efeito dos juros compostos."
        ),
    },
    {
        "pergunta": "O que é crédito e como usá-lo de forma consciente?",
        "paginas_relevantes": {32, 35, 42, 43},
        "resposta_referencia": (
            "Crédito é uma fonte adicional de recursos obtidos de terceiros "
            "que possibilita antecipar consumo. Para usá-lo de forma "
            "consciente, deve-se planejar dentro do orçamento, usar para "
            "fins essenciais, evitar endividamento excessivo e reduzir "
            "despesas proporcionalmente em caso de perda de renda."
        ),
    },
]

print(f"Dataset de avaliação criado com {len(dataset_avaliacao)} perguntas.")


# ----------------------------------------------------------------
# 14.2  EXECUTAR RECUPERAÇÃO PARA CADA PERGUNTA
# ----------------------------------------------------------------
# Roda o pipeline de busca (embedding + ChromaDB) e guarda as
# páginas recuperadas e as distâncias de cada pergunta.
# ----------------------------------------------------------------

K = 5  # nº de chunks recuperados (igual ao usado no RAG)

resultados_recuperacao = []

for item in dataset_avaliacao:
    pergunta = item["pergunta"]

    # Embedding da pergunta (prefixo "query:" exigido pelo e5)
    vetor = modelo_embeddings.encode(
        "query: " + pergunta,
        normalize_embeddings=True,
    )

    # Busca no ChromaDB
    busca = colecao.query(
        query_embeddings=[vetor.tolist()],
        n_results=K,
        include=["metadatas", "distances"],
    )

    resultados_recuperacao.append({
        "pergunta": pergunta,
        "paginas_recuperadas": [m["pagina"] for m in busca["metadatas"][0]],
        "distancias": busca["distances"][0],
        "paginas_relevantes": item["paginas_relevantes"],
    })

print(f"Recuperação executada para {len(resultados_recuperacao)} perguntas (K={K}).\n")
for r in resultados_recuperacao:
    print(f"Pergunta: {r['pergunta'][:60]}...")
    print(f"  Relevantes:  {sorted(r['paginas_relevantes'])}")
    print(f"  Recuperadas: {r['paginas_recuperadas']}")
    print()


# ----------------------------------------------------------------
# 14.3  MÉTRICAS DE RECUPERAÇÃO
# ----------------------------------------------------------------
# Definições adotadas (o ChromaDB devolve CHUNKS, mas o ground truth
# está anotado em PÁGINAS, então cada métrica deixa claro o nível):
#
#   Recall@K    → nível de PÁGINA: páginas relevantes distintas que
#                 aparecem entre os K chunks / total de páginas relevantes.
#   Precision@K → nível de CHUNK: chunks (dos K) que vêm de uma página
#                 relevante / K.
#   MRR         → 1 / posição do primeiro chunk de página relevante.
#   NDCG@K      → ganho binário por página; cada página relevante só
#                 pontua na PRIMEIRA vez que aparece, para que chunks
#                 duplicados da mesma página não inflem o DCG.
#   Únicas      → quantas páginas distintas vieram nos K chunks
#                 (mostra o quanto as duplicatas ocupam o contexto).
# ----------------------------------------------------------------

def dedup(paginas):
    """Remove páginas repetidas mantendo a ordem de recuperação."""
    vistos, saida = set(), []
    for p in paginas:
        if p not in vistos:
            vistos.add(p)
            saida.append(p)
    return saida


def recall_at_k(recuperadas, relevantes, k):
    if not relevantes:
        return 0.0
    return len(set(recuperadas[:k]) & relevantes) / len(relevantes)


def precision_at_k(recuperadas, relevantes, k):
    if k == 0:
        return 0.0
    return sum(1 for p in recuperadas[:k] if p in relevantes) / k


def mrr_score(recuperadas, relevantes):
    for i, p in enumerate(recuperadas, start=1):
        if p in relevantes:
            return 1.0 / i
    return 0.0


def ndcg_at_k(recuperadas, relevantes, k):
    vistos, dcg = set(), 0.0
    for i, p in enumerate(recuperadas[:k], start=1):
        if p in relevantes and p not in vistos:
            dcg += 1.0 / math.log2(i + 1)
            vistos.add(p)
    n_rel = min(len(relevantes), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, n_rel + 1))
    return dcg / idcg if idcg > 0 else 0.0


# --- Calcular as métricas por pergunta ---
linhas_metricas = []

for r in resultados_recuperacao:
    rec = r["paginas_recuperadas"]
    rel = r["paginas_relevantes"]

    linhas_metricas.append({
        "pergunta": r["pergunta"][:50],
        "recall": recall_at_k(rec, rel, K),
        "precision": precision_at_k(rec, rel, K),
        "mrr": mrr_score(rec, rel),
        "ndcg": ndcg_at_k(rec, rel, K),
        "unicas": len(dedup(rec[:K])),
    })

# --- Tabela ---
print("=" * 84)
print("  MÉTRICAS DE RECUPERAÇÃO (Retrieval)")
print("=" * 84)
print(f"{'Pergunta':<50} {'Rec@5':>6} {'Prec@5':>6} {'MRR':>6} {'NDCG':>6} {'Únicas':>7}")
print("-" * 84)
for l in linhas_metricas:
    print(
        f"{l['pergunta']:<50} {l['recall']:>6.3f} {l['precision']:>6.3f} "
        f"{l['mrr']:>6.3f} {l['ndcg']:>6.3f} {l['unicas']:>5}/{K}"
    )
print("-" * 84)

med_recall = float(np.mean([l["recall"] for l in linhas_metricas]))
med_precision = float(np.mean([l["precision"] for l in linhas_metricas]))
med_mrr = float(np.mean([l["mrr"] for l in linhas_metricas]))
med_ndcg = float(np.mean([l["ndcg"] for l in linhas_metricas]))
med_unicas = float(np.mean([l["unicas"] for l in linhas_metricas]))

print(
    f"{'MÉDIA':<50} {med_recall:>6.3f} {med_precision:>6.3f} "
    f"{med_mrr:>6.3f} {med_ndcg:>6.3f} {med_unicas:>7.1f}"
)
print("=" * 84)

In [0]:
# ============================================================
# 14.4  MÉTRICAS DE GERAÇÃO (BLEU e ROUGE)
# ============================================================
# Para cada pergunta:
#   1. Recupera os chunks e monta o contexto (guardado para o juiz).
#   2. Gera a resposta com o mesmo pipeline da Etapa 13.
#   3. Remove as citações "[Página X do PDF]" e compara com a
#      referência usando BLEU-4 e ROUGE-L.
#
# Correções em relação à versão anterior:
#   - O gpt-oss-20b é um modelo de raciocínio: parte do max_tokens é
#     consumida pelo raciocínio interno. Com limite baixo, o texto
#     final pode vir vazio. Por isso o limite foi aumentado.
#   - O content pode vir como lista de blocos (raciocínio + texto);
#     extrair_texto() pega só o texto final.
#   - As citações de página são removidas antes do BLEU/ROUGE, pois
#     não existem na referência e só diluem a sobreposição.
#   - Resposta vazia vira NaN (falha), não 0 (qualidade ruim).
#   - O contexto de cada pergunta é guardado e reutilizado no juiz,
#     garantindo que ele avalie exatamente o que o gerador recebeu.
# ============================================================

import re
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

try:
    word_tokenize("teste", language="portuguese")
except LookupError:
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)

MODELO_LLM = "databricks-gpt-oss-20b"
MAX_TOKENS_GERACAO = 2000


# ----------------------------------------------------------------
# 14.4a  Funções auxiliares
# ----------------------------------------------------------------

def extrair_texto(mensagem):
    """Devolve só o texto final da resposta do LLM.

    Em modelos de raciocínio, message.content pode ser uma string ou
    uma lista de blocos (ex.: um bloco de raciocínio e um de texto).
    """
    conteudo = mensagem.content
    if conteudo is None:
        return ""
    if isinstance(conteudo, str):
        return conteudo

    partes = []
    for bloco in conteudo:
        if isinstance(bloco, dict):
            tipo, texto = bloco.get("type"), bloco.get("text")
        else:
            tipo, texto = getattr(bloco, "type", None), getattr(bloco, "text", None)
        if tipo == "text" and texto:
            partes.append(texto)
    return "".join(partes)


PADRAO_CITACAO = re.compile(r"\[\s*P[áa]ginas?[^\]]*\]", re.IGNORECASE)


def limpar_citacoes(texto):
    """Remove marcações como [Página 42 do PDF] antes das métricas."""
    sem_citacao = PADRAO_CITACAO.sub("", texto)
    return re.sub(r"\s{2,}", " ", sem_citacao).strip()


def tokenizar_pt(texto):
    return word_tokenize(texto.lower(), language="portuguese")


def calcular_bleu(referencia, gerada):
    """BLEU-4 com suavização (method1)."""
    ref_tokens = tokenizar_pt(referencia)
    ger_tokens = tokenizar_pt(gerada)
    return sentence_bleu(
        [ref_tokens], ger_tokens,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=SmoothingFunction().method1,
    )


def calcular_rouge_l(referencia, gerada):
    """ROUGE-L (F1) a partir da maior subsequência comum (LCS)."""
    ref_tokens = tokenizar_pt(referencia)
    ger_tokens = tokenizar_pt(gerada)

    m, n = len(ref_tokens), len(ger_tokens)
    if m == 0 or n == 0:
        return 0.0

    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i - 1] == ger_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs = dp[m][n]
    if lcs == 0:
        return 0.0
    recall = lcs / m
    precision = lcs / n
    return (2 * precision * recall) / (precision + recall)


def media_valida(valores):
    """Média ignorando NaN. Devolve (média, quantidade de valores válidos)."""
    validos = [v for v in valores if not np.isnan(v)]
    if not validos:
        return float("nan"), 0
    return float(np.mean(validos)), len(validos)


def montar_contexto(pergunta):
    """Recupera os K chunks e monta o contexto com referências de página."""
    vetor = modelo_embeddings.encode(
        "query: " + pergunta,
        normalize_embeddings=True,
    )
    busca = colecao.query(
        query_embeddings=[vetor.tolist()],
        n_results=K,
        include=["documents", "metadatas"],
    )
    trechos = []
    for j, (texto, metadata) in enumerate(
        zip(busca["documents"][0], busca["metadatas"][0]),
        start=1,
    ):
        trechos.append(
            f"[TRECHO {j} | Página {metadata['pagina']} do PDF]\n{texto}"
        )
    return "\n\n".join(trechos)


# ----------------------------------------------------------------
# 14.4b  Gerar respostas do RAG
# ----------------------------------------------------------------

system_prompt_av = """
Você é um assistente de educação financeira.
Responda em português do Brasil, utilizando
exclusivamente as informações do contexto fornecido.

Regras:
1. Não invente informações.
2. Não utilize conhecimentos externos.
3. Fundamente as afirmações nos trechos recuperados.
4. Cite as páginas do PDF que sustentam a resposta.
5. Utilize referências como [Página 42 do PDF].
6. Se não houver informações suficientes,
   declare explicitamente essa limitação.
7. Não apresente recomendações como se fossem
   do Banco Central sem evidência no contexto.
"""


def gerar_resposta(contexto, pergunta, tentativas=3):
    for t in range(tentativas):
        resp = client.chat.completions.create(
            model=MODELO_LLM,
            messages=[
                {"role": "system", "content": system_prompt_av},
                {
                    "role": "user",
                    "content": f"CONTEXTO:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
                },
            ],
            max_tokens=MAX_TOKENS_GERACAO,
            temperature=0.0 if t == 0 else 0.2,
        )
        texto = extrair_texto(resp.choices[0].message).strip()
        if texto:
            return texto
        print(
            f"    (geração: tentativa {t + 1} vazia, "
            f"finish_reason={resp.choices[0].finish_reason})"
        )
    return ""


print("Gerando respostas do RAG para cada pergunta...\n")

respostas_rag = []

for i, item in enumerate(dataset_avaliacao, start=1):
    pergunta = item["pergunta"]
    contexto = montar_contexto(pergunta)
    resposta_texto = gerar_resposta(contexto, pergunta)

    if resposta_texto:
        resposta_limpa = limpar_citacoes(resposta_texto)
        bleu = calcular_bleu(item["resposta_referencia"], resposta_limpa)
        rouge = calcular_rouge_l(item["resposta_referencia"], resposta_limpa)
    else:
        bleu, rouge = float("nan"), float("nan")

    respostas_rag.append({
        "pergunta": pergunta,
        "contexto": contexto,
        "resposta_rag": resposta_texto,
        "resposta_ref": item["resposta_referencia"],
        "bleu": bleu,
        "rouge_l": rouge,
    })

    status = "FALHA (resposta vazia)" if not resposta_texto else f"BLEU={bleu:.4f}  ROUGE-L={rouge:.4f}"
    print(f"  [{i}/{len(dataset_avaliacao)}] {status}")

# --- Tabela ---
print("\n" + "=" * 68)
print("  MÉTRICAS DE GERAÇÃO (BLEU-4 e ROUGE-L, citações removidas)")
print("=" * 68)
print(f"{'Pergunta':<48} {'BLEU-4':>8} {'ROUGE-L':>8}")
print("-" * 68)
for r in respostas_rag:
    bleu_txt = f"{r['bleu']:>8.4f}" if not np.isnan(r["bleu"]) else f"{'—':>8}"
    rouge_txt = f"{r['rouge_l']:>8.4f}" if not np.isnan(r["rouge_l"]) else f"{'—':>8}"
    print(f"{r['pergunta'][:48]:<48} {bleu_txt} {rouge_txt}")
print("-" * 68)

med_bleu, n_bleu = media_valida([r["bleu"] for r in respostas_rag])
med_rouge, _ = media_valida([r["rouge_l"] for r in respostas_rag])
print(f"{'MÉDIA (n=' + str(n_bleu) + '/' + str(len(respostas_rag)) + ')':<48} {med_bleu:>8.4f} {med_rouge:>8.4f}")
print("=" * 68)

# Exemplo para inspeção
print("\n--- EXEMPLO (Pergunta 1) ---")
print(f"Referência:\n{respostas_rag[0]['resposta_ref']}\n")
print(f"RAG:\n{respostas_rag[0]['resposta_rag'][:600]}")

In [0]:
# ============================================================
# 14.5  LLM-AS-A-JUDGE
# ============================================================
# O LLM avalia cada resposta em três critérios (0 a 5):
#   GROUNDEDNESS        → a resposta se baseia só no contexto?
#   RELEVANCE           → responde diretamente à pergunta?
#   FACTUAL_CONSISTENCY → as afirmações batem com o contexto?
#
# Correções em relação à versão anterior:
#   - O juiz recebe o contexto COMPLETO usado na geração (antes era
#     cortado em 2000 caracteres e ele não via parte da evidência).
#   - max_tokens maior, pelo mesmo motivo da 14.4 (modelo de raciocínio).
#   - Só aceita a avaliação se as três notas forem extraídas; senão,
#     tenta de novo.
#   - Falha vira NaN (não 0) e a média informa quantas perguntas
#     foram efetivamente avaliadas.
#
# Limitação a registrar no relatório: o juiz é o mesmo modelo que
# gerou as respostas (viés de autoavaliação).
# ============================================================

MAX_TOKENS_JUIZ = 2000
CRITERIOS = ["GROUNDEDNESS", "RELEVANCE", "FACTUAL_CONSISTENCY"]


def avaliar_com_llm(pergunta, contexto, resposta, tentativas=5):
    prompt_juiz = f"""Você é um avaliador especialista. Analise a resposta gerada
por um sistema RAG com base nos critérios abaixo.

CONTEXTO RECUPERADO:
{contexto}

PERGUNTA DO USUÁRIO:
{pergunta}

RESPOSTA DO RAG:
{resposta}

Avalie cada critério com uma nota inteira de 0 a 5 (0 = péssimo, 5 = excelente):

1. GROUNDEDNESS: A resposta se baseia exclusivamente no contexto? Há alucinação?
2. RELEVANCE: A resposta responde diretamente à pergunta?
3. FACTUAL_CONSISTENCY: As afirmações são coerentes com o contexto?

Responda EXATAMENTE neste formato, sem nenhum texto adicional:
GROUNDEDNESS: <nota>
RELEVANCE: <nota>
FACTUAL_CONSISTENCY: <nota>"""

    for t in range(tentativas):
        resp = client.chat.completions.create(
            model=MODELO_LLM,
            messages=[
                {"role": "system", "content": "Você é um avaliador imparcial de sistemas de IA."},
                {"role": "user", "content": prompt_juiz},
            ],
            max_tokens=MAX_TOKENS_JUIZ,
            temperature=0.0 if t == 0 else 0.2,
        )
        texto = extrair_texto(resp.choices[0].message).strip()

        notas = {}
        for criterio in CRITERIOS:
            m = re.search(rf"{criterio}\s*[:=\-]\s*\**\s*([0-5])", texto.upper())
            if m:
                notas[criterio.lower()] = int(m.group(1))

        if len(notas) == len(CRITERIOS):
            return notas

        motivo = "vazia" if not texto else "fora do formato"
        print(
            f"    (juiz: tentativa {t + 1} {motivo}, "
            f"finish_reason={resp.choices[0].finish_reason})"
        )

    return {}


print("Executando LLM-as-a-Judge para cada resposta...\n")

avaliacoes_juiz = []

for i, r in enumerate(respostas_rag, start=1):
    if r["resposta_rag"]:
        notas = avaliar_com_llm(r["pergunta"], r["contexto"], r["resposta_rag"])
    else:
        notas = {}  # não há o que avaliar se a geração falhou

    avaliacao = {
        "pergunta": r["pergunta"][:45],
        "groundedness": float(notas.get("groundedness", float("nan"))),
        "relevance": float(notas.get("relevance", float("nan"))),
        "factual_consistency": float(notas.get("factual_consistency", float("nan"))),
    }
    avaliacoes_juiz.append(avaliacao)

    if notas:
        print(
            f"  [{i}/{len(respostas_rag)}] Ground={notas['groundedness']}  "
            f"Relev={notas['relevance']}  Factual={notas['factual_consistency']}"
        )
    else:
        print(f"  [{i}/{len(respostas_rag)}] NÃO AVALIADA")


def fmt_nota(v):
    return f"{int(v):>7}" if not np.isnan(v) else f"{'—':>7}"


# --- Tabela ---
print("\n" + "=" * 72)
print("  LLM-AS-A-JUDGE (notas 0-5)")
print("=" * 72)
print(f"{'Pergunta':<45} {'Ground':>7} {'Relev':>7} {'Factual':>7}")
print("-" * 72)
for a in avaliacoes_juiz:
    print(
        f"{a['pergunta']:<45} {fmt_nota(a['groundedness'])} "
        f"{fmt_nota(a['relevance'])} {fmt_nota(a['factual_consistency'])}"
    )
print("-" * 72)

med_ground, n_juiz = media_valida([a["groundedness"] for a in avaliacoes_juiz])
med_relev, _ = media_valida([a["relevance"] for a in avaliacoes_juiz])
med_factual, _ = media_valida([a["factual_consistency"] for a in avaliacoes_juiz])

print(f"{'MÉDIA (n=' + str(n_juiz) + '/' + str(len(avaliacoes_juiz)) + ')':<45} {med_ground:>7.1f} {med_relev:>7.1f} {med_factual:>7.1f}")
print("=" * 72)
if n_juiz < len(avaliacoes_juiz):
    print(f"Atenção: {len(avaliacoes_juiz) - n_juiz} pergunta(s) não avaliada(s); a média não representa o conjunto completo.")

In [0]:
# ============================================================
# 14.6  RESUMO CONSOLIDADO + RECOMENDAÇÕES
# ============================================================
# Todos os números vêm das variáveis calculadas acima (nada fixo
# no texto), então o resumo continua certo se o pipeline mudar.
# ============================================================

n_total = len(dataset_avaliacao)

print("=" * 75)
print("  RESUMO CONSOLIDADO — AVALIAÇÃO DO RAG")
print(f"  {n_total} perguntas | K = {K} | modelo de embeddings: {NOME_MODELO}")
print("=" * 75)
print()

print("RECUPERAÇÃO (Retrieval) — escala 0 a 1:")
print(f"  Recall@{K}      = {med_recall:.3f}   (páginas relevantes encontradas)")
print(f"  Precision@{K}   = {med_precision:.3f}   (chunks vindos de página relevante)")
print(f"  MRR           = {med_mrr:.3f}   (posição do 1º chunk relevante)")
print(f"  NDCG@{K}        = {med_ndcg:.3f}   (qualidade do ranking, sem duplicatas)")
print(f"  Páginas únicas = {med_unicas:.1f}/{K} por consulta")
print()

print(f"GERAÇÃO (Generation) — escala 0 a 1 (n={n_bleu}/{n_total}):")
print(f"  BLEU-4        = {med_bleu:.4f}  (n-gramas vs. referência)")
print(f"  ROUGE-L       = {med_rouge:.4f}  (subsequência comum vs. referência)")
print()

print(f"LLM-AS-A-JUDGE — escala 0 a 5 (n={n_juiz}/{n_total}):")
print(f"  Groundedness       = {med_ground:.1f}/5")
print(f"  Relevance          = {med_relev:.1f}/5")
print(f"  Factual Consist.   = {med_factual:.1f}/5")
print()

# --- Distribuição das distâncias (base para escolher um limiar) ---
dist_rel, dist_irrel = [], []
for r in resultados_recuperacao:
    for pag, dist in zip(r["paginas_recuperadas"], r["distancias"]):
        (dist_rel if pag in r["paginas_relevantes"] else dist_irrel).append(dist)

print("DISTÂNCIAS COSSENO DOS CHUNKS RECUPERADOS:")
for nome, valores in [("Página relevante", dist_rel), ("Página irrelevante", dist_irrel)]:
    if valores:
        print(
            f"  {nome:<19} n={len(valores):>2}  "
            f"mín={np.min(valores):.3f}  mediana={np.median(valores):.3f}  máx={np.max(valores):.3f}"
        )
print("  (Se as faixas se sobrepõem, um filtro só por distância não separa relevantes de irrelevantes.)")
print("=" * 75)

# ----------------------------------------------------------------
# RECOMENDAÇÕES
# ----------------------------------------------------------------

recomendacoes = [
    {
        "metrica": f"Recall@{K} ({med_recall:.3f})",
        "problema": f"Em média, {(1 - med_recall) * 100:.0f}% das páginas relevantes não são recuperadas.",
        "acao": (
            "Aumentar K para 8–10 e/ou testar busca híbrida (BM25 + embeddings), "
            "que captura termos exatos que a busca semântica pode perder. "
            "Testar também a recuperação com múltiplas consultas (Etapa 15)."
        ),
    },
    {
        "metrica": f"Precision@{K} ({med_precision:.3f})",
        "problema": "Parte dos chunks recuperados vem de páginas fora do ground truth.",
        "acao": (
            "Testar reranking com cross-encoder multilíngue sobre os top-20. "
            "Antes de usar filtro por distância, conferir na tabela de distâncias "
            "acima se existe um limiar que separe relevantes de irrelevantes. "
            "Reconferir o ground truth no PDF: páginas vizinhas podem conter a "
            "resposta e estar faltando na anotação."
        ),
    },
    {
        "metrica": f"MRR ({med_mrr:.3f}) e NDCG@{K} ({med_ndcg:.3f})",
        "problema": (
            f"Relevantes nem sempre aparecem no topo, e só {med_unicas:.1f} de {K} "
            "chunks vêm de páginas distintas."
        ),
        "acao": (
            "Recuperar mais candidatos (ex.: 15) e deduplicar por página antes de "
            "montar o contexto, ou usar MMR (Maximal Marginal Relevance) para "
            "diversificar os trechos."
        ),
    },
    {
        "metrica": f"BLEU-4 ({med_bleu:.4f}) e ROUGE-L ({med_rouge:.4f})",
        "problema": "Pouca sobreposição literal com as respostas de referência.",
        "acao": (
            "Esperado: as referências são resumos curtos e as respostas do RAG são "
            "mais longas e parafraseadas. Tratar como métrica secundária e "
            "complementar com BERTScore ou similaridade de embeddings entre "
            "resposta e referência."
        ),
    },
    {
        "metrica": "Groundedness / Factual Consistency",
        "problema": (
            f"Notas avaliadas em {n_juiz} de {n_total} perguntas, com o mesmo modelo "
            "atuando como gerador e juiz."
        ),
        "acao": (
            "Usar um modelo diferente como juiz, quando disponível, e validar uma "
            "amostra das notas manualmente. Incluir perguntas sem resposta no "
            "documento para testar se o sistema declara a limitação."
        ),
    },
    {
        "metrica": "Modelo de embeddings",
        "problema": f"{NOME_MODELO} (384 dimensões) é leve, mas menos preciso.",
        "acao": (
            "Comparar com intfloat/multilingual-e5-base (768 dim), "
            "intfloat/multilingual-e5-large (1024 dim) ou BAAI/bge-m3 (1024 dim), "
            "todos multilíngues. Contrapartida: mais memória e tempo de encoding."
        ),
    },
    {
        "metrica": "Chunking",
        "problema": "Chunks de 800 caracteres podem misturar assuntos ou cortar explicações.",
        "acao": (
            "Comparar chunk_size=500/overlap=100 com a configuração atual "
            "(800/120), usando separators=['\\n\\n', '\\n', '. ', ' '] para "
            "respeitar parágrafos e frases."
        ),
    },
]

print("\nRECOMENDAÇÕES DE MELHORIA:")
print("=" * 75)
for rec in recomendacoes:
    print(f"\n• {rec['metrica']}")
    print(f"  Problema: {rec['problema']}")
    print(f"  Ação:     {rec['acao']}")
print("\n" + "=" * 75)